# Ticket 9: period report for finance sign-off

Consolidating the final numbers `reports/period_2024-01.md` needs to
report, and cross-checking every table this report reads (`recon_period_summary`,
`recon_mismatch`, `recon_reversal_pairs`, `dq_violations`) against each
other one last time before writing anything finance-facing.

In [1]:
import duckdb

con = duckdb.connect("../warehouse.duckdb", read_only=True)

In [2]:
# headline reconciliation total: stg vs fact, all 502 accounts
con.execute("""
    SELECT COUNT(*), ROUND(SUM(stg_local_total),2), ROUND(SUM(fact_local_total),2), ROUND(SUM(local_amount_gap),2)
    FROM recon_period_summary
""").fetchall()

[(502, 97144587.12, 97144587.12, 0.0)]

**502 accounts, stg total = fact total = 97,144,587.12, gap = 0.00.** A
naive read of this number says the period reconciles perfectly. It
doesn't say the number itself is trustworthy: mission 06 already found
20 of those documents carry a source-data defect (a document total
broadcast across every debit line) that both `stg_gl` and `fact_gl_line`
inherit identically, which is exactly why the gap between them is zero.
This is the single most important thing the report has to get right:
showing "gap = 0.00" without the caveat would tell finance something
false.

In [3]:
# mismatch bucket + cause table, count and amount, per issue #9's explicit ask
con.execute("SELECT bucket, cause, COUNT(*), ROUND(SUM(gap),2) FROM recon_mismatch GROUP BY 1,2 ORDER BY 1,2").fetchall()

[('intentionally_excluded', 'opening_balance', 17, -0.01),
 ('intentionally_excluded', 'post_close', 200, 578374.47),
 ('missing_in_fact', 'unbalanced_document', 2, 0.0)]

`opening_balance`: 17 rows, net -0.01 (one 17-line brought-forward
document, individually large lines that net to ~$0 in aggregate, not a
residual to chase). `post_close`: 200 rows, net 578,374.47.
`unbalanced_document`: 2 rows, net 0.00 (the excluded document's two
lines exactly offset each other, which is exactly why it's flagged
unbalanced on debit/credit despite netting to zero on amount).

In [4]:
# reversal pairs summary, per issue #9's explicit ask for a reversal-pair count
con.execute("""
    SELECT COUNT(*), SUM(cross_period::int), SUM(is_net_zero::int), SUM((NOT is_net_zero)::int)
    FROM recon_reversal_pairs
""").fetchall()

[(1025, 326, 940, 85)]

1,025 pairs, 326 cross-period, 940 valid, 85 flagged as planted
fake-reversal anomalies (mission 07).

In [5]:
# non-blocking quality-gate findings, for the report's data-quality context section
con.execute("SELECT check_name, blocking, COUNT(*) FROM dq_violations GROUP BY 1,2 ORDER BY 1").fetchall()

[('catch_all_account', False, 70),
 ('local_amount_imbalance', False, 23),
 ('unbalanced_document', True, 1),
 ('unmapped_account', False, 15)]

## What I've got

Every number the report needs is already built and cross-checked:
reconciliation (mission 06), mismatch bucket/cause (mission 08),
reversal pairs (mission 07), quality-gate findings (mission 05). The one
open question mission 09 has to resolve isn't a number, it's a decision:
does "gap = 0.00, 0% unknown" - true, and clean by every check this
project runs - qualify as a period finance can sign, given the known
$97,144,587.1 `local_amount` defect neither side of the comparison can
see? See the mission's Human approvals.